## Import Libraries

In [1]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F
from tqdm import tqdm

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.8.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce GTX 1650


## Configuration and Paths

In [2]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Paths
ROOT_DIR = "/home/vishn/Documents/5/ML/Code/Project/output/"
INPUT_DIR = os.path.join(ROOT_DIR, "input/")
FEATURES_DIR = os.path.join(ROOT_DIR, "FEATURES_DIR/")
PARTAWARE_DIR = os.path.join(ROOT_DIR, "PartAwarePooling/")
os.makedirs(PARTAWARE_DIR, exist_ok=True)

# Files
TRAIN_CSV = os.path.join(INPUT_DIR, "train.csv")
VAL_CSV = os.path.join(INPUT_DIR, "validation.csv")
TRAIN_FEATURES = os.path.join(FEATURES_DIR, "train_features.npy")
VAL_FEATURES = os.path.join(FEATURES_DIR, "val_features.npy")

print(f"\nConfiguration:")
print(f"  Input CSVs: {INPUT_DIR}")
print(f"  Pre-extracted features: {FEATURES_DIR}")
print(f"  Output directory: {PARTAWARE_DIR}")

Using device: cuda

Configuration:
  Input CSVs: /home/vishn/Documents/5/ML/Code/Project/output/input/
  Pre-extracted features: /home/vishn/Documents/5/ML/Code/Project/output/FEATURES_DIR/
  Output directory: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/


## Load Pre-Extracted Features (From Feature_extraction.ipynb)

In [3]:
print("Loading pre-extracted ResNet-50 features...")
train_features = np.load(TRAIN_FEATURES)  # [312186, 2048]
val_features = np.load(VAL_FEATURES)      # [52490, 2048]

print("Loading metadata CSVs...")
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

# Parse segmentation (needed for part masks)
print("Parsing segmentation data...")
train_df['segmentation'] = train_df['segmentation'].apply(lambda x: json.loads(x.replace("'", "\"")))
val_df['segmentation'] = val_df['segmentation'].apply(lambda x: json.loads(x.replace("'", "\"")))

print(f"\n✓ Loaded:")
print(f"  Train: {train_features.shape[0]:,} samples, features shape: {train_features.shape}")
print(f"  Val: {val_features.shape[0]:,} samples, features shape: {val_features.shape}")

Loading pre-extracted ResNet-50 features...
Loading metadata CSVs...
Parsing segmentation data...

✓ Loaded:
  Train: 312,186 samples, features shape: (312186, 2048)
  Val: 52,490 samples, features shape: (52490, 2048)


## Part-Aware Pooling Functions (Modular, Functional)

In [4]:
def normalize_masks(part_masks):
    """Normalize part masks to [0, 1] range."""
    eps = 1e-8
    B, K, H, W = part_masks.shape
    masks_flat = part_masks.view(B, K, -1)
    mask_min = masks_flat.min(dim=2, keepdim=True)[0].unsqueeze(-1)
    mask_max = masks_flat.max(dim=2, keepdim=True)[0].unsqueeze(-1)
    normalized_masks = (part_masks - mask_min) / (mask_max - mask_min + eps)
    return torch.clamp(normalized_masks, 0.0, 1.0)


def part_aware_pooling(feature_map, part_masks):
    """
    Weighted spatial pooling per part: f_k = sum(mask_k * feature_map) / sum(mask_k)
    
    Args:
        feature_map: [B, C, H, W]
        part_masks: [B, K, H, W]
    Returns:
        part_features: [B, K, C]
    """
    B, C, H, W = feature_map.shape
    _, K, _, _ = part_masks.shape
    eps = 1e-8
    part_features = torch.zeros(B, K, C, device=feature_map.device, dtype=feature_map.dtype)
    
    for k in range(K):
        mask_k = part_masks[:, k, :, :].unsqueeze(1)  # [B, 1, H, W]
        weighted_features = feature_map * mask_k
        numerator = weighted_features.sum(dim=(2, 3))  # [B, C]
        denominator = mask_k.sum(dim=(2, 3))  # [B, 1]
        part_features[:, k, :] = numerator / (denominator + eps)
    
    return part_features


def fuse_part_features(part_features, use_attention=False, attention_weights=None):
    """Fuse K part features into single representation."""
    B, K, C = part_features.shape
    
    if use_attention and attention_weights is not None:
        if attention_weights.dim() == 1:
            weights = attention_weights.view(1, K, 1)
        else:
            weights = attention_weights.unsqueeze(2)
        weights = F.softmax(weights, dim=1)
        fused_feature = (part_features * weights).sum(dim=1)
    else:
        fused_feature = part_features.mean(dim=1)  # Simple average
    
    return fused_feature


def classify(fused_feature, num_classes, classifier_weights=None, classifier_bias=None):
    """Apply linear classifier."""
    B, C = fused_feature.shape
    
    if classifier_weights is None:
        classifier_weights = torch.randn(num_classes, C, device=fused_feature.device) * 0.01
    if classifier_bias is None:
        classifier_bias = torch.zeros(num_classes, device=fused_feature.device)
    
    logits = torch.matmul(fused_feature, classifier_weights.t()) + classifier_bias
    return logits


def forward_pipeline(feature_map, part_masks, num_classes, use_attention=False, 
                     attention_weights=None, classifier_weights=None, classifier_bias=None):
    """Complete Part-Aware Pooling pipeline."""
    normalized_masks = normalize_masks(part_masks)
    part_features = part_aware_pooling(feature_map, normalized_masks)
    fused_feature = fuse_part_features(part_features, use_attention, attention_weights)
    logits = classify(fused_feature, num_classes, classifier_weights, classifier_bias)
    return logits, fused_feature, part_features

print("✓ Part-Aware Pooling functions defined")

✓ Part-Aware Pooling functions defined


## Generate Part Masks from Segmentation

In [5]:
def create_part_mask(segmentation, img_size=224, target_size=7, num_parts=5):
    """Create part masks from segmentation polygons (divide into horizontal strips)."""
    full_mask = np.zeros((img_size, img_size), dtype=np.float32)
    
    for polygon in segmentation:
        points = np.array(polygon).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(full_mask, [points], 1.0)
    
    # Resize to target resolution
    mask_resized = cv2.resize(full_mask, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
    
    # Divide into horizontal parts
    part_masks = np.zeros((num_parts, target_size, target_size), dtype=np.float32)
    strip_height = target_size / num_parts
    
    for k in range(num_parts):
        y_start = int(k * strip_height)
        y_end = int((k + 1) * strip_height)
        part_mask = mask_resized.copy()
        part_mask[:y_start, :] = 0
        part_mask[y_end:, :] = 0
        part_masks[k] = part_mask
    
    return torch.from_numpy(part_masks).float()


def generate_all_part_masks(df, num_parts=5, target_size=7):
    """Generate part masks for entire dataset."""
    all_masks = []
    for idx in tqdm(range(len(df)), desc="Generating part masks"):
        segmentation = df.iloc[idx]['segmentation']
        masks = create_part_mask(segmentation, target_size=target_size, num_parts=num_parts)
        all_masks.append(masks)
    return torch.stack(all_masks)

print("✓ Part mask generation functions defined")

✓ Part mask generation functions defined


## Process Full Dataset: Apply Part-Aware Pooling

**Processing entire dataset**: 312k train + 52k validation samples

In [6]:
NUM_PARTS = 5
SPATIAL_SIZE = 7

print("="*80)
print("LOADING OR GENERATING PART MASKS")
print("="*80)

# Check if part masks already exist
train_masks_file = os.path.join(PARTAWARE_DIR, 'train_part_masks.pt')
val_masks_file = os.path.join(PARTAWARE_DIR, 'val_part_masks.pt')

if os.path.exists(train_masks_file) and os.path.exists(val_masks_file):
    print("\n✓ Found existing part masks, loading...")
    train_part_masks = torch.load(train_masks_file)
    val_part_masks = torch.load(val_masks_file)
    print(f"✓ Loaded train masks: {train_part_masks.shape}")
    print(f"✓ Loaded val masks: {val_part_masks.shape}")
else:
    print("\n⚠️  Part masks not found, generating...")
    print("[1/2] Generating training part masks...")
    train_part_masks = generate_all_part_masks(train_df, NUM_PARTS, SPATIAL_SIZE)
    print(f"✓ Train masks: {train_part_masks.shape}")
    
    print("\n[2/2] Generating validation part masks...")
    val_part_masks = generate_all_part_masks(val_df, NUM_PARTS, SPATIAL_SIZE)
    print(f"✓ Val masks: {val_part_masks.shape}")
    
    # Save for future use
    torch.save(train_part_masks, train_masks_file)
    torch.save(val_part_masks, val_masks_file)
    print(f"\n✓ Part masks saved to {PARTAWARE_DIR}")

LOADING OR GENERATING PART MASKS

✓ Found existing part masks, loading...
✓ Loaded train masks: torch.Size([312186, 5, 7, 7])
✓ Loaded val masks: torch.Size([52490, 5, 7, 7])


In [12]:
print("\n"+"="*80)
print("MEMORY-EFFICIENT PROCESSING SETUP")
print("="*80)

# DON'T create full spatial tensors in memory! 
# Instead, we'll convert features to spatial format in small batches during processing
print("\n✓ Will process features in batches to avoid memory overload")
print(f"  Train samples: {len(train_features):,}")
print(f"  Val samples: {len(val_features):,}")
print(f"  Memory strategy: Convert [N,2048]→[N,2048,7,7] in batches during processing")


MEMORY-EFFICIENT PROCESSING SETUP

✓ Will process features in batches to avoid memory overload


NameError: name 'train_features' is not defined

In [7]:
print("\n"+"="*80)
print("APPLYING PART-AWARE POOLING (BATCH-WISE SAVING)")
print("="*80)

BATCH_SIZE = 256  # Process in small batches
BATCH_DIR = os.path.join(PARTAWARE_DIR, 'batches')
os.makedirs(BATCH_DIR, exist_ok=True)

def process_and_save_batches(features_np, part_masks, batch_size, prefix):
    """
    Process dataset in batches and save each batch immediately to disk.
    This prevents memory accumulation.
    """
    n_samples = len(features_np)
    batch_files_part = []
    batch_files_fused = []
    
    for i in tqdm(range(0, n_samples, batch_size), desc=f"{prefix} batches"):
        end_idx = min(i + batch_size, n_samples)
        batch_num = i // batch_size
        
        # Convert batch to spatial format on-the-fly
        batch_features = torch.from_numpy(features_np[i:end_idx]).float()
        batch_spatial = batch_features.view(-1, 2048, 1, 1).expand(-1, 2048, SPATIAL_SIZE, SPATIAL_SIZE)
        batch_spatial = batch_spatial.to(device)
        
        # Load corresponding masks
        batch_masks = part_masks[i:end_idx].to(device)
        
        # Apply part-aware pooling
        normalized_masks = normalize_masks(batch_masks)
        part_feats = part_aware_pooling(batch_spatial, normalized_masks)
        fused_feats = fuse_part_features(part_feats, use_attention=False)
        
        # Save batch to disk immediately
        part_file = os.path.join(BATCH_DIR, f'{prefix}_part_batch_{batch_num}.npy')
        fused_file = os.path.join(BATCH_DIR, f'{prefix}_fused_batch_{batch_num}.npy')
        
        np.save(part_file, part_feats.cpu().numpy())
        np.save(fused_file, fused_feats.cpu().numpy())
        
        batch_files_part.append(part_file)
        batch_files_fused.append(fused_file)
        
        # Aggressive memory cleanup
        del batch_features, batch_spatial, batch_masks, normalized_masks, part_feats, fused_feats
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return batch_files_part, batch_files_fused, n_samples

# Process training set
print(f"\n[1/2] Processing training set ({len(train_features):,} samples)...")
train_part_files, train_fused_files, train_n = process_and_save_batches(
    train_features, train_part_masks, BATCH_SIZE, 'train'
)
print(f"✓ Saved {len(train_part_files)} training batches")

# Clear memory
del train_features, train_part_masks
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Process validation set
print(f"\n[2/2] Processing validation set ({len(val_features):,} samples)...")
val_part_files, val_fused_files, val_n = process_and_save_batches(
    val_features, val_part_masks, BATCH_SIZE, 'val'
)
print(f"✓ Saved {len(val_part_files)} validation batches")

# Clear memory
del val_features, val_part_masks
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✓ All batches processed and saved!")


APPLYING PART-AWARE POOLING (BATCH-WISE SAVING)

[1/2] Processing training set (312,186 samples)...


train batches: 100%|██████████| 1220/1220 [01:12<00:00, 16.76it/s]


✓ Saved 1220 training batches

[2/2] Processing validation set (52,490 samples)...


val batches: 100%|██████████| 206/206 [00:11<00:00, 17.34it/s]

✓ Saved 206 validation batches

✓ All batches processed and saved!


## Save Output Features

## Merge Batch Files into Final Features

In [8]:
print("\n"+"="*80)
print("MERGING BATCHES INTO FINAL FEATURES")
print("="*80)

def merge_batches_memmap(batch_files, output_file, expected_shape):
    """
    Merge batch files into a single memory-mapped array.
    This avoids loading all data into RAM at once.
    """
    # Create memory-mapped array
    memmap_array = np.memmap(output_file, dtype='float32', mode='w+', shape=expected_shape)
    
    # Copy each batch
    current_idx = 0
    for batch_file in tqdm(batch_files, desc=f"Merging to {os.path.basename(output_file)}"):
        batch_data = np.load(batch_file)
        batch_size = batch_data.shape[0]
        memmap_array[current_idx:current_idx + batch_size] = batch_data
        current_idx += batch_size
        del batch_data  # Free memory
    
    # Flush to disk
    memmap_array.flush()
    del memmap_array
    return output_file

# Merge training features
print("\n[1/4] Merging training part features...")
train_part_output = os.path.join(PARTAWARE_DIR, 'train_part_features.npy')
merge_batches_memmap(train_part_files, train_part_output, (train_n, NUM_PARTS, 2048))
print(f"✓ Saved: {train_part_output}")

print("\n[2/4] Merging training fused features...")
train_fused_output = os.path.join(PARTAWARE_DIR, 'train_fused_features.npy')
merge_batches_memmap(train_fused_files, train_fused_output, (train_n, 2048))
print(f"✓ Saved: {train_fused_output}")

# Merge validation features
print("\n[3/4] Merging validation part features...")
val_part_output = os.path.join(PARTAWARE_DIR, 'val_part_features.npy')
merge_batches_memmap(val_part_files, val_part_output, (val_n, NUM_PARTS, 2048))
print(f"✓ Saved: {val_part_output}")

print("\n[4/4] Merging validation fused features...")
val_fused_output = os.path.join(PARTAWARE_DIR, 'val_fused_features.npy')
merge_batches_memmap(val_fused_files, val_fused_output, (val_n, 2048))
print(f"✓ Saved: {val_fused_output}")

print("\n✓ All features merged successfully!")

# Clean up batch files to save disk space
print("\nCleaning up batch files...")
import shutil
shutil.rmtree(BATCH_DIR)
print(f"✓ Deleted temporary batch directory: {BATCH_DIR}")


MERGING BATCHES INTO FINAL FEATURES

[1/4] Merging training part features...


Merging to train_part_features.npy: 100%|██████████| 1220/1220 [00:46<00:00, 26.34it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/train_part_features.npy

[2/4] Merging training fused features...


Merging to train_fused_features.npy: 100%|██████████| 1220/1220 [00:10<00:00, 119.78it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/train_fused_features.npy

[3/4] Merging validation part features...


Merging to val_part_features.npy: 100%|██████████| 206/206 [00:07<00:00, 28.26it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/val_part_features.npy

[4/4] Merging validation fused features...


Merging to val_fused_features.npy: 100%|██████████| 206/206 [00:01<00:00, 134.48it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/val_fused_features.npy

✓ All features merged successfully!

Cleaning up batch files...
✓ Deleted temporary batch directory: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/batches


## Final Summary: Input → Process → Output

## Process Test Dataset (Grid-Based Masks)

Since the test dataset doesn't have segmentation data, we'll use uniform grid-based part masks (dividing the spatial map into 5 equal horizontal strips).

In [10]:
# Load test dataset
TEST_CSV = os.path.join(INPUT_DIR, "test.csv")
TEST_FEATURES = os.path.join(FEATURES_DIR, "test_features.npy")

print("Loading test dataset...")
test_features = np.load(TEST_FEATURES)
test_df = pd.read_csv(TEST_CSV)

print(f"✓ Test: {test_features.shape[0]:,} samples, features shape: {test_features.shape}")

# Create uniform grid-based masks for test set (no segmentation available)
def create_grid_masks(n_samples, num_parts=5, spatial_size=7):
    """
    Create uniform grid-based masks by dividing spatial map into horizontal strips.
    Used for test set where segmentation is not available.
    """
    masks = torch.zeros(n_samples, num_parts, spatial_size, spatial_size)
    strip_height = spatial_size / num_parts
    
    for k in range(num_parts):
        y_start = int(k * strip_height)
        y_end = int((k + 1) * strip_height)
        masks[:, k, y_start:y_end, :] = 1.0
    
    return masks

print(f"\nGenerating grid-based masks for test set (no segmentation available)...")
test_part_masks = create_grid_masks(len(test_features), NUM_PARTS, SPATIAL_SIZE)
print(f"✓ Test masks: {test_part_masks.shape}")

Loading test dataset...
✓ Test: 62,629 samples, features shape: (62629, 2048)

Generating grid-based masks for test set (no segmentation available)...
✓ Test masks: torch.Size([62629, 5, 7, 7])


In [13]:
# Process test set with batch-wise saving
print("\n"+"="*80)
print("PROCESSING TEST DATASET")
print("="*80)

print(f"\nProcessing test set ({len(test_features):,} samples)...")
test_part_files, test_fused_files, test_n = process_and_save_batches(
    test_features, test_part_masks, BATCH_SIZE, 'test'
)
print(f"✓ Saved {len(test_part_files)} test batches")

# Clear memory
del test_features, test_part_masks
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Merge test batches
print("\n"+"="*80)
print("MERGING TEST BATCHES")
print("="*80)

print("\n[1/2] Merging test part features...")
test_part_output = os.path.join(PARTAWARE_DIR, 'test_part_features.npy')
merge_batches_memmap(test_part_files, test_part_output, (test_n, NUM_PARTS, 2048))
print(f"✓ Saved: {test_part_output}")

print("\n[2/2] Merging test fused features...")
test_fused_output = os.path.join(PARTAWARE_DIR, 'test_fused_features.npy')
merge_batches_memmap(test_fused_files, test_fused_output, (test_n, 2048))
print(f"✓ Saved: {test_fused_output}")

print("\n✓ Test dataset processing complete!")


PROCESSING TEST DATASET

Processing test set (62,629 samples)...


test batches: 100%|██████████| 245/245 [00:15<00:00, 15.96it/s]


✓ Saved 245 test batches

MERGING TEST BATCHES

[1/2] Merging test part features...


Merging to test_part_features.npy: 100%|██████████| 245/245 [00:05<00:00, 48.11it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/test_part_features.npy

[2/2] Merging test fused features...


Merging to test_fused_features.npy: 100%|██████████| 245/245 [00:01<00:00, 202.91it/s]


✓ Saved: /home/vishn/Documents/5/ML/Code/Project/output/PartAwarePooling/test_fused_features.npy

✓ Test dataset processing complete!
